# 03 - Gold Layer - Product Dimension

Create a business-ready Product Dimension from the validated Silver product data.

**Source:** `end-to-end_pipeline.silver.products`

**Target:** `end-to-end_pipeline.gold.dim_product`

**Model Role:** Dimension Table

**Business Key:** `product_id`

**Approach:** Profile → Inspect → Transform → Validate

**Purpose:** Provide descriptive product attributes for analyzing sales by product, category, subcategory, brand, supplier, product status, and launch date.

#Cell 1 - Profile Silver Product Data

Description:
Confirm that the Silver Products table is ready to become a Gold dimension. This checks product-key uniqueness, row count, and availability of the main business attributes needed for reporting.


In [0]:
%sql

-- ============================================================
-- CELL 1: PROFILE SILVER PRODUCTS FOR GOLD MODELING
-- Purpose: Confirm dimension grain, key uniqueness,
--          and availability of business attributes
-- ============================================================

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT product_id) AS distinct_product_ids,
    COUNT(*) - COUNT(DISTINCT product_id) AS duplicate_product_ids,

    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END)
        AS null_product_ids,

    COUNT(DISTINCT category)
        AS categories,

    COUNT(DISTINCT subcategory)
        AS subcategories,

    COUNT(DISTINCT brand)
        AS brands,

    COUNT(DISTINCT product_status)
        AS product_statuses,

    MIN(launch_date)
        AS earliest_launch_date,

    MAX(launch_date)
        AS latest_launch_date,

    SUM(CASE WHEN list_price IS NULL THEN 1 ELSE 0 END)
        AS null_list_prices,

    MIN(list_price) AS min_list_price,

    MAX(list_price) AS max_list_price,

    SUM(CASE WHEN standard_cost IS NULL THEN 1 ELSE 0 END)
        AS null_standard_costs,

    MIN(standard_cost) AS min_standard_cost,

    MAX(standard_cost) AS max_standard_cost

FROM `end-to-end_pipeline`.silver.products;

total_rows,distinct_product_ids,duplicate_product_ids,null_product_ids,categories,subcategories,brands,product_statuses,earliest_launch_date,latest_launch_date,null_list_prices,min_list_price,max_list_price,null_standard_costs,min_standard_cost,max_standard_cost
248,248,0,0,5,16,8,2,2018-01-22,2024-10-24,0,9.55,1246.88,0,5.18,697.19


#Cell 2 - Inspect Product Business Attributes

Description:
Review the main descriptive product attributes that will be exposed through the Gold dimension. This helps confirm that the dimension supports meaningful business analysis by category, subcategory, brand, and status.

In [0]:
%sql

-- ============================================================
-- CELL 2: INSPECT PRODUCT BUSINESS ATTRIBUTES
-- Purpose: Review product distribution across business groups
-- ============================================================

SELECT
    category,
    subcategory,
    product_status,
    COUNT(*) AS product_count

FROM `end-to-end_pipeline`.silver.products

GROUP BY
    category,
    subcategory,
    product_status

ORDER BY
    category,
    subcategory,
    product_status;

category,subcategory,product_status,product_count
Accessories,Cables,Active,10
Accessories,Cables,Discontinued,1
Accessories,Headsets,Active,17
Accessories,Headsets,Discontinued,1
Accessories,Keyboards,Active,17
Accessories,Keyboards,Discontinued,2
Accessories,Mice,Active,19
Accessories,Mice,Discontinued,1
Electronics,Laptops,Active,14
Electronics,Laptops,Discontinued,2


#Cell 3 - Transform Silver → Gold Product Dimension

Description:
Create the Gold Product Dimension at one row per product.

This step exposes the descriptive attributes needed for analysis. Silver-layer cleaning is not repeated.

product_id remains the business key that will later connect dim_product to fact_sales.

In [0]:
%sql

-- ============================================================
-- CELL 3: CREATE GOLD PRODUCT DIMENSION
-- Grain: One row per product
-- Business Key: product_id
-- ============================================================

CREATE OR REPLACE TABLE `end-to-end_pipeline`.gold.dim_product
COMMENT 'Business-ready product dimension. One row per product. Source: silver.products. Used by dashboards, Genie, and fact_sales joins.'
AS

SELECT
    product_id,
    product_name,
    category,
    subcategory,
    brand,
    supplier_id,
    list_price,
    standard_cost,
    launch_date,
    YEAR(launch_date) AS launch_year,
    product_status

FROM `end-to-end_pipeline`.silver.products;

COMMENT ON COLUMN `end-to-end_pipeline`.gold.dim_product.product_id IS 'Unique product business key; joins to fact_sales.product_id';
COMMENT ON COLUMN `end-to-end_pipeline`.gold.dim_product.product_name IS 'Full product display name';
COMMENT ON COLUMN `end-to-end_pipeline`.gold.dim_product.category IS 'Product category';
COMMENT ON COLUMN `end-to-end_pipeline`.gold.dim_product.subcategory IS 'Product subcategory';
COMMENT ON COLUMN `end-to-end_pipeline`.gold.dim_product.brand IS 'Product brand';
COMMENT ON COLUMN `end-to-end_pipeline`.gold.dim_product.supplier_id IS 'Supplier identifier';
COMMENT ON COLUMN `end-to-end_pipeline`.gold.dim_product.list_price IS 'Product reference list price';
COMMENT ON COLUMN `end-to-end_pipeline`.gold.dim_product.standard_cost IS 'Product standard cost';
COMMENT ON COLUMN `end-to-end_pipeline`.gold.dim_product.launch_date IS 'Date the product was launched';
COMMENT ON COLUMN `end-to-end_pipeline`.gold.dim_product.launch_year IS 'Derived: year of product launch, useful for cohort analysis in dashboards';
COMMENT ON COLUMN `end-to-end_pipeline`.gold.dim_product.product_status IS 'Product status (e.g., Active, Discontinued)';

#Cell 4 - Validate Gold Product Dimension

Description:
Validate that the Product Dimension maintains one row per product, contains valid business attributes, and preserves the expected Silver-to-Gold row relationship.

In [0]:
%sql

-- ============================================================
-- CELL 4: VALIDATE GOLD PRODUCT DIMENSION
-- Purpose: Confirm dimension grain, key integrity,
--          and Silver → Gold completeness
-- ============================================================

WITH validation AS (

    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT product_id)
            AS distinct_product_ids,

        COUNT(*) - COUNT(DISTINCT product_id)
            AS duplicate_product_ids,

        SUM(
            CASE
                WHEN product_id IS NULL THEN 1
                ELSE 0
            END
        ) AS null_product_ids,

        SUM(
            CASE
                WHEN product_name IS NULL THEN 1
                ELSE 0
            END
        ) AS null_product_names,

        SUM(
            CASE
                WHEN category IS NULL THEN 1
                ELSE 0
            END
        ) AS null_categories,

        SUM(
            CASE
                WHEN product_status IS NULL THEN 1
                ELSE 0
            END
        ) AS null_product_status,

        SUM(
            CASE
                WHEN brand IS NULL THEN 1
                ELSE 0
            END
        ) AS null_brands,

        SUM(
            CASE
                WHEN subcategory IS NULL THEN 1
                ELSE 0
            END
        ) AS null_subcategories,

        SUM(
            CASE
                WHEN supplier_id IS NULL THEN 1
                ELSE 0
            END
        ) AS null_supplier_ids

    FROM `end-to-end_pipeline`.gold.dim_product
),

source_check AS (

    SELECT
        COUNT(*) AS silver_rows

    FROM `end-to-end_pipeline`.silver.products
)

SELECT
    v.*,
    s.silver_rows,

    CASE
        WHEN v.total_rows = s.silver_rows
            AND v.distinct_product_ids = v.total_rows
            AND v.duplicate_product_ids = 0
            AND v.null_product_ids = 0
            AND v.null_product_names = 0
            AND v.null_categories = 0
            AND v.null_product_status = 0
            AND v.null_brands = 0
            AND v.null_subcategories = 0
            AND v.null_supplier_ids = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END AS validation_status

FROM validation v
CROSS JOIN source_check s;

total_rows,distinct_product_ids,duplicate_product_ids,null_product_ids,null_product_names,null_categories,null_product_status,null_brands,null_subcategories,null_supplier_ids,silver_rows,validation_status
248,248,0,0,0,0,0,0,0,0,248,PASS
